In [65]:
"""Functions for finding downstream vertices and alternative edges."""

import networkx as nx
import pandas as pd


class IDNotFoundError(Exception):
    pass


class InputLengthDoesNotMatchError(Exception):
    pass


class IDNotUniqueError(Exception):
    pass


class GraphNotFullyConnectedError(Exception):
    pass


class GraphCycleError(Exception):
    pass


class EdgeAlreadyDisabledError(Exception):
    pass


class InvalidEdgeTableError(ValueError):
    pass


class GraphProcessor:
    """
    Processes an undirected graph where enabled edges must form a spanning tree
    (fully connected, no cycles). A source vertex is designated as the root.

    Supports:
    - find_downstream_vertices: all vertices on the far side of an edge from the source.
    - find_alternative_edges: disabled edges that can replace a given enabled edge.
    """

    EDGE_COLUMNS = ("edge_id", "from_vertex", "to_vertex", "enabled")

    def __init__(
        self,
        edge_table: pd.DataFrame,
        source_vertex_id: int,
        vertex_ids: list[int] | None = None,
    ) -> None:
        """Initialize an undirected graph from a table with one row per edge."""
        if not isinstance(edge_table, pd.DataFrame):
            raise InvalidEdgeTableError("edge_table must be a pandas DataFrame.")

        missing_columns = set(self.EDGE_COLUMNS) - set(edge_table.columns)
        if missing_columns:
            missing = ", ".join(sorted(missing_columns))
            raise InvalidEdgeTableError(f"edge_table is missing required columns: {missing}.")

        self.edge_table = edge_table.loc[:, self.EDGE_COLUMNS].copy()
        if self.edge_table.isna().any().any():
            raise InvalidEdgeTableError("edge_table cannot contain missing values.")
        if not self.edge_table.empty and not pd.api.types.is_bool_dtype(self.edge_table["enabled"]):
            raise InvalidEdgeTableError("The enabled column must contain boolean values.")

        edge_ids = self.edge_table["edge_id"].tolist()
        if len(edge_ids) != len(set(edge_ids)):
            raise IDNotUniqueError("Duplicate edge IDs.")

        if vertex_ids is None:
            vertex_ids = list(
                dict.fromkeys(self.edge_table["from_vertex"].tolist() + self.edge_table["to_vertex"].tolist())
            )
        elif len(vertex_ids) != len(set(vertex_ids)):
            raise IDNotUniqueError("Duplicate vertex IDs.")

        vertex_set = set(vertex_ids)
        edge_vertex_id_pairs = list(
            self.edge_table.loc[:, ["from_vertex", "to_vertex"]].itertuples(index=False, name=None)
        )
        for u, v in edge_vertex_id_pairs:
            if u not in vertex_set or v not in vertex_set:
                raise IDNotFoundError("Edge references unknown vertex.")

        if source_vertex_id not in vertex_set:
            raise IDNotFoundError("Invalid source_vertex_id.")

        self.source = source_vertex_id
        self.edge_pairs = dict(zip(edge_ids, edge_vertex_id_pairs, strict=True))
        self.edge_enabled = dict(zip(edge_ids, self.edge_table["enabled"], strict=True))

        # Build graph with only enabled edges
        self._graph = nx.Graph()
        self._graph.add_nodes_from(vertex_ids)
        enabled_edge_count = 0
        for eid, (u, v), en in zip(edge_ids, edge_vertex_id_pairs, self.edge_table["enabled"], strict=True):
            if en:
                enabled_edge_count += 1
                self._graph.add_edge(u, v, edge_id=eid)

        if not nx.is_connected(self._graph):
            raise GraphNotFullyConnectedError("Graph is not fully connected.")
        if enabled_edge_count != len(vertex_ids) - 1:
            raise GraphCycleError("Graph contains a cycle.")

        self._tree = nx.bfs_tree(self._graph, self.source)

    def find_downstream_vertices(self, edge_id: int) -> list[int]:
        """
        Given an edge id, return all the vertices which are in the downstream of the edge,
            with respect to the source vertex.
            Including the downstream vertex of the edge itself!

        Only enabled edges should be taken into account in the analysis.
        If the given edge_id is a disabled edge, it should return empty list.
        If the given edge_id does not exist, it should raise IDNotFoundError.


        For example, given the following graph (all edges enabled):

            vertex_0 (source) --edge_1-- vertex_2 --edge_3-- vertex_4

        Call find_downstream_vertices with edge_id=1 will return [2, 4]
        Call find_downstream_vertices with edge_id=3 will return [4]

        Args:
            edge_id: edge id to be searched

        Returns:
            A list of all downstream vertices.
        """
        if edge_id not in self.edge_pairs:
            raise IDNotFoundError(f"Edge {edge_id} not found.")
        if not self.edge_enabled[edge_id]:
            return []

        u, v = self.edge_pairs[edge_id]
        downstream = v if self._tree.has_edge(u, v) else u
        return [downstream] + list(nx.descendants(self._tree, downstream))

    def find_alternative_edges(self, disabled_edge_id: int) -> list[int]:
        """
        Given an enabled edge, do the following analysis:
            If the edge is going to be disabled,
                which (currently disabled) edge can be enabled to ensure
                that the graph is again fully connected and acyclic?
            Return a list of all alternative edges.
        If the disabled_edge_id is not a valid edge id, it should raise IDNotFoundError.
        If the disabled_edge_id is already disabled, it should raise EdgeAlreadyDisabledError.
        If there are no alternative to make the graph fully connected again, it should return empty list.


        For example, given the following graph:

        vertex_0 (source) --edge_1(enabled)-- vertex_2 --edge_9(enabled)-- vertex_10
                 |                               |
                 |                           edge_7(disabled)
                 |                               |
                 -----------edge_3(enabled)-- vertex_4
                 |                               |
                 |                           edge_8(disabled)
                 |                               |
                 -----------edge_5(enabled)-- vertex_6

        Call find_alternative_edges with disabled_edge_id=1 will return [7]
        Call find_alternative_edges with disabled_edge_id=3 will return [7, 8]
        Call find_alternative_edges with disabled_edge_id=5 will return [8]
        Call find_alternative_edges with disabled_edge_id=9 will return []

        Args:
            disabled_edge_id: edge id (which is currently enabled) to be disabled

        Returns:
            A list of alternative edge ids.
        """
        if disabled_edge_id not in self.edge_pairs:
            raise IDNotFoundError(f"Edge {disabled_edge_id} not found.")
        if not self.edge_enabled[disabled_edge_id]:
            raise EdgeAlreadyDisabledError(f"Edge {disabled_edge_id} is already disabled.")

        downstream_vertices = set(self.find_downstream_vertices(disabled_edge_id))

        return [
            edge_id
            for edge_id, (u, v) in self.edge_pairs.items()
            if not self.edge_enabled[edge_id] and ((u in downstream_vertices) != (v in downstream_vertices))
        ]


# 📘 Assignment 1 – Graph Processing


## 🧩 Overview
In this assignment, we implemented a **GraphProcessor** class that models an undirected graph.

### 🎯 Goals
- Build a graph from tabular data
- Ensure enabled edges form a **spanning tree**
- Implement:
  - Downstream vertex detection
  - Alternative edge detection

This notebook explains the logic, shows the implementation, and demonstrates results.



## 🌳 Graph Concept
A valid graph must:
- Be **fully connected**
- Contain **no cycles**

This makes it a **spanning tree**.

Why?
- Ensures unique paths
- Enables efficient traversal


## 🏗️ Core Implementation (Explanation)


### Initialization
The GraphProcessor:
- Validates input data
- Builds a graph using NetworkX
- Keeps only enabled edges
- Checks:
  - Connectivity
  - No cycles

### BFS Tree
A BFS tree is created from the source node.
This defines direction in the graph.


In [66]:
import pandas as pd

from power_system_simulation.graph_processing import GraphProcessor


def edge_table(edge_ids, edge_pairs, enabled):
    return pd.DataFrame(
        {
            "edge_id": edge_ids,
            "from_vertex": [p[0] for p in edge_pairs],
            "to_vertex": [p[1] for p in edge_pairs],
            "enabled": enabled,
        }
    )


## 📊 Example Graphs

In [67]:
# Linear graph
linear_graph = GraphProcessor(
    edge_table([1, 3], [(0, 2), (2, 4)], [True, True]),
    source_vertex_id=0
)

# Branched graph
branched_graph = GraphProcessor(
    edge_table(
        [1, 3, 5, 7, 8, 9],
        [(0, 2), (0, 4), (0, 6), (2, 4), (4, 6), (2, 10)],
        [True, True, True, False, False, True],
    ),
    source_vertex_id=0
)



## 🔍 Downstream Vertices

### Idea
When an edge is followed away from the source:
- The graph splits conceptually
- We return all nodes on the "far" side

### Algorithm
1. Check edge validity
2. Use BFS tree to determine direction
3. Use descendants to collect nodes


### ✅ Results

In [68]:
print('Edge 1 downstream:', linear_graph.find_downstream_vertices(1))
print('Edge 3 downstream:', linear_graph.find_downstream_vertices(3))
print('Branched edge 1 downstream:', branched_graph.find_downstream_vertices(1))


Edge 1 downstream: [2, 4]
Edge 3 downstream: [4]
Branched edge 1 downstream: [2, 10]



## 🔄 Alternative Edges

### Idea
If an enabled edge is removed:
- The graph splits into two parts
- We find disabled edges that reconnect these parts

### Algorithm
1. Get downstream vertices
2. Check disabled edges
3. Select edges connecting both partitions


### ✅ Results

In [69]:
print('Alt edges for 1:', branched_graph.find_alternative_edges(1))
print('Alt edges for 3:', branched_graph.find_alternative_edges(3))
print('Alt edges for 9:', branched_graph.find_alternative_edges(9))


Alt edges for 1: [7]
Alt edges for 3: [7, 8]
Alt edges for 9: []



## ⚠️ Error Handling

The implementation ensures robustness using custom exceptions:
- Invalid IDs
- Cycles
- Disconnected graphs
- Incorrect data


In [70]:
from power_system_simulation.graph_processing import IDNotFoundError

try:
    linear_graph.find_downstream_vertices(99)
except IDNotFoundError as e:
    print('Error caught:', e)


Error caught: Edge 99 not found.



## 🧪 Testing

We validated:
- Correct outputs
- Edge cases
- Error conditions

Examples:
- Invalid edge
- Disabled edge
- No alternative edges



## ✅ Conclusion

This assignment demonstrates:
- Graph theory fundamentals
- Efficient traversal using BFS
- Robust validation and error handling

### 🚀 Key Takeaways
- Trees simplify graph logic
- BFS defines direction
- Partitioning enables alternative edge detection


In [71]:
from pathlib import Path

import numpy as np
import pandas as pd
from power_grid_model import ComponentType, PowerGridModel
from power_grid_model.utils import json_deserialize
from power_grid_model.validation import errors_to_string, validate_batch_data, validate_input_data


class InputDataValidationError(ValueError):
    pass


class BatchDataValidationError(ValueError):
    pass


class ProfileTimestampMismatchError(ValueError):
    pass


class ProfileLoadIDMismatchError(ValueError):
    pass


class TimeIndexLengthError(ValueError):
    pass


def read_json_file(file_path):
    return Path(file_path).read_text()


def deserialize_pgm_json(json_data):
    return json_deserialize(json_data)


def create_pgm(input_data):
    errors = validate_input_data(input_data)
    if errors:
        raise InputDataValidationError(errors_to_string(errors, name="input data", details=True))
    return PowerGridModel(input_data)


def read_load_profile(file_path):
    return pd.read_parquet(file_path)


def validate_load_profile(active_profile, reactive_profile):
    if active_profile.empty or reactive_profile.empty:
        raise ValueError("Load profiles cannot be empty.")
    if not active_profile.index.equals(reactive_profile.index):
        raise ProfileTimestampMismatchError("Active and reactive profiles have different timestamps.")
    if not active_profile.index.is_unique or not active_profile.index.is_monotonic_increasing:
        raise ProfileTimestampMismatchError("Load profile timestamps must be unique and sorted.")
    if not active_profile.columns.equals(reactive_profile.columns):
        raise ProfileLoadIDMismatchError("Active and reactive profiles have different load IDs.")
    if not active_profile.columns.is_unique:
        raise ProfileLoadIDMismatchError("Load profile IDs must be unique.")


def create_load_batch_update(active_profile, reactive_profile):
    validate_load_profile(active_profile, reactive_profile)

    load_ids = np.tile(
        active_profile.columns.to_numpy(dtype=np.int32),
        (len(active_profile), 1),
    )

    return {
        ComponentType.sym_load: {
            "id": load_ids,
            "p_specified": active_profile.to_numpy(),
            "q_specified": reactive_profile.to_numpy(),
        }
    }


def run_batch_power_flow(input_data, batch_update):
    errors = validate_batch_data(input_data, batch_update)
    if errors:
        raise BatchDataValidationError(errors_to_string(errors, name="batch update", details=True))
    model = PowerGridModel(input_data)
    return model.calculate_power_flow(update_data=batch_update)


def aggregate_node_voltage_results(results, time_index):
    node_results = results[ComponentType.node]
    if len(node_results) != len(time_index):
        raise TimeIndexLengthError("The number of timestamps does not match the number of result batches.")

    rows = []
    for i, timestamp in enumerate(time_index):
        ids = node_results[i]["id"]
        u_pu = node_results[i]["u_pu"]

        max_idx = np.argmax(u_pu)
        min_idx = np.argmin(u_pu)

        rows.append(
            {
                "timestamp": timestamp,
                "max_u_pu": u_pu[max_idx],
                "max_u_pu_node_id": ids[max_idx],
                "min_u_pu": u_pu[min_idx],
                "min_u_pu_node_id": ids[min_idx],
            }
        )
    return pd.DataFrame(rows).set_index("timestamp")


def aggregate_line_results(results, time_index):
    line_results = results[ComponentType.line]
    if len(line_results) != len(time_index):
        raise TimeIndexLengthError("The number of timestamps does not match the number of result batches.")

    line_ids = line_results[0]["id"]

    rows = []
    for line_index, line_id in enumerate(line_ids):
        loading = line_results["loading"][:, line_index]
        max_idx = np.argmax(loading)
        min_idx = np.argmin(loading)
        loss_w = line_results["p_from"][:, line_index] + line_results["p_to"][:, line_index]
        hours = (time_index - time_index[0]).total_seconds() / 3600
        energy_loss_kwh = np.trapezoid(loss_w, hours) / 1000
        rows.append(
            {
                "line_id": line_id,
                "max_loading_pu": loading[max_idx],
                "max_loading_timestamp": time_index[max_idx],
                "min_loading_pu": loading[min_idx],
                "min_loading_timestamp": time_index[min_idx],
                "energy_loss_kwh": energy_loss_kwh,
            }
        )

    return pd.DataFrame(rows).set_index("line_id")


# 📘 Assignment 2 – Power Grid Simulation

## 🧩 Overview
In this assignment, we simulate an electrical power grid over time using computational models.

A power grid is a network of nodes and connections that transport electricity from generation to consumers.

### 🎯 Goals
- Validate grid input data
- Process load profiles
- Run batch simulations
- Extract insights

---

# 🌐 Power Flow Concept

## What is Power Flow?
Power flow analysis determines how electricity moves through a network.

It calculates:
- Voltages at nodes
- Power flows in lines
- Network losses

👉 It answers: *Can the grid safely operate under given conditions?*

# ⚙️ Step 1 – Input Handling

We load JSON data and convert it into a simulation model.

### Why validation matters:
- Prevents invalid networks
- Ensures safe simulation
- Detects structural issues early

In [72]:
from pathlib import Path
import numpy as np
import pandas as pd
from power_grid_model import ComponentType, PowerGridModel
from power_grid_model.utils import json_deserialize
from power_grid_model.validation import errors_to_string, validate_batch_data, validate_input_data

## ⚠️ Custom Exceptions
Clear error handling improves robustness.

In [73]:
class InputDataValidationError(ValueError): pass
class BatchDataValidationError(ValueError): pass
class ProfileTimestampMismatchError(ValueError): pass
class ProfileLoadIDMismatchError(ValueError): pass
class TimeIndexLengthError(ValueError): pass

# 📊 Step 2 – Load Profiles

Load profiles define how demand changes over time.

### Requirements:
- Matching timestamps
- Matching load IDs
- Sorted and unique data

### Why this matters:
Incorrect data → incorrect physics → unreliable results

In [74]:
def read_load_profile(path):
    return pd.read_parquet(path)

def validate_load_profile(a, r):
    if a.empty or r.empty: raise ValueError('empty')
    if not a.index.equals(r.index): raise ProfileTimestampMismatchError()
    if not a.columns.equals(r.columns): raise ProfileLoadIDMismatchError()

# 🔁 Step 3 – Batch Processing

We transform profiles into a simulation-ready format.

### Key idea:
Run many timesteps efficiently in one call.

In [75]:
def create_load_batch_update(a,r):
    validate_load_profile(a,r)
    ids=np.tile(a.columns.to_numpy(dtype=np.int32),(len(a),1))
    return {ComponentType.sym_load:{'id':ids,'p_specified':a.to_numpy(),'q_specified':r.to_numpy()}}

# ⚡ Step 4 – Simulation

The model computes voltages and flows at each timestep.

In [76]:
def run_batch_power_flow(input_data,batch):
    errors=validate_batch_data(input_data,batch)
    if errors: raise BatchDataValidationError()
    return PowerGridModel(input_data).calculate_power_flow(update_data=batch)

# 📈 Step 5 – Node Analysis

We extract:
- Max voltage
- Min voltage

### Why:
Voltage limits determine grid stability.

In [77]:
def aggregate_node_voltage_results(results,time_index):
    nodes=results[ComponentType.node]
    rows=[]
    for i,t in enumerate(time_index):
        ids=nodes[i]['id']; u=nodes[i]['u_pu']
        rows.append({'timestamp':t,'max':u.max(),'min':u.min()})
    return pd.DataFrame(rows).set_index('timestamp')

# 📉 Step 6 – Line Analysis

We compute:
- Loading extremes
- Energy loss

### Trapezoidal Rule
We approximate energy by integrating power over time.

In [78]:
def aggregate_line_results(results,time_index):
    line=results[ComponentType.line]
    hours=(time_index-time_index[0]).total_seconds()/3600
    ids=line[0]['id']
    rows=[]
    for i,lid in enumerate(ids):
        loading=line['loading'][:,i]
        loss=line['p_from'][:,i]+line['p_to'][:,i]
        rows.append({'line_id':lid,'energy_loss_kwh':np.trapezoid(loss,hours)/1000})
    return pd.DataFrame(rows).set_index('line_id')

# ✅ Example Pipeline

In [79]:
# Example usage (requires data files)
input_data=deserialize_pgm_json(read_json_file('data/input_network_data.json'))
active=read_load_profile('data/active_power_profile.parquet')
reactive=read_load_profile('data/reactive_power_profile.parquet')
batch=create_load_batch_update(active,reactive)
results=run_batch_power_flow(input_data,batch)

# ✅ Conclusion

### Key Takeaways
- Power grids behave as networks
- Time-series simulation reveals dynamic behavior
- Validation is essential
- Numerical integration enables energy calculations

# 📘 Assignment 3 – LV Grid Analytics

## 🧩 Overview
In this assignment, we build a **complete low-voltage grid analytics pipeline** by combining:
- Graph theory (Assignment 1)
- Power flow simulation (Assignment 2)

This allows us to analyze realistic grid scenarios such as EV integration, optimization, and failures.

---

# 🌐 LV Grid Concept
A low-voltage (LV) grid is modeled as a network of nodes (houses) and lines.

### Key characteristics:
- Radial (tree-like) during operation
- Can contain extra disconnected lines (ring structure)
- Contains exactly one transformer and one source

### Why this matters
- Tree structure simplifies power flow
- Ring structure improves reliability

---

# ✅ Step 1 – Grid & Data Validation

## 🧠 Idea
Before simulation, we must ensure all inputs are valid.

### What we validate:
- PGM structure correctness
- One transformer and one source
- Feeder correctness
- Load & EV profile consistency

### Why this is important
Invalid inputs → incorrect physics → meaningless results

---

In [80]:
# Example imports
from power_system_simulation.lv_grid_validation import validate_grid, validate_feeders, validate_load_profiles, validate_ev_profiles

# 🚗 Step 2 – EV Penetration

## 🧠 Idea
We simulate the effect of electric vehicles on the grid.

### How it works
1. Divide loads per feeder
2. Randomly assign EVs to houses
3. Add EV charging profiles to loads

### Key constraints
- Equal distribution across feeders
- Each profile used only once
- Reproducibility via random seed

### Insight
EVs significantly increase peak demand and stress the network

---

In [81]:
# Example usage
# results = apply_ev_penetration(grid, feeder_ids, load_p, load_q, ev_profiles, 0.2, seed=42)

# ⚡ Step 3 – Tap Optimization

## 🧠 Idea
Transformers can adjust voltage using tap positions.

### Goal
Find the best tap position by:
- Minimizing energy loss OR
- Minimizing voltage deviation

### Algorithm
1. Try each tap position
2. Run full time-series simulation
3. Evaluate objective
4. Select best result

### Insight
Small tap changes can significantly improve performance

---

In [82]:
# Example usage
# best_tap = optimize_tap_position(grid, load_p, load_q, criterion='loss')

# 🔄 Step 4 – N-1 Contingency Analysis

## 🧠 Idea
We simulate failure of a line (outage).

### Process
1. Disconnect a line
2. Find alternative edges (Assignment 1 logic)
3. Reconnect using alternatives
4. Run simulation

### Output
- Worst loading in scenario
- Line causing it
- Timestamp

### Insight
This ensures grid reliability under failures

---

In [83]:
# Example usage
# n1_table = calculate_n_minus_1(grid, load_p, load_q, outage_line_id=8)

# 🧪 Testing Strategy

## Coverage
We validate:
- Correct outputs
- Edge cases
- Error handling

### Examples
- Invalid feeder IDs
- Not enough houses per feeder
- Duplicate EV profiles
- Invalid tap criterion

### Key insight
Robust tests ensure correctness under all scenarios

---

# ✅ Full Workflow

```python
# Step 1: Validate
graph = validate_grid(grid)
validate_feeders(grid, feeder_ids)
validate_load_profiles(grid, load_p, load_q)
validate_ev_profiles(grid, load_p, ev_profiles)

# Step 2: EV Simulation
ev_results = apply_ev_penetration(grid, feeder_ids, load_p, load_q, ev_profiles, 0.3, seed=1)

# Step 3: Optimization
best_tap = optimize_tap_position(grid, load_p, load_q, 'loss')

# Step 4: N-1
n1_results = calculate_n_minus_1(grid, load_p, load_q, outage_line_id=8)
```

---

# ✅ Conclusion

## 🚀 Key Takeaways
- LV grid analytics combines multiple domains
- Graph theory enables topology analysis
- Simulation captures real-world dynamics
- Optimization improves efficiency
- N-1 ensures reliability

## 🎯 Final Insight
This assignment integrates **modeling, simulation, optimization, and resilience analysis** into one powerful workflow.